# Begin

In [3]:
import os # @launchit.collect
import sys # @launchit.collect
import copy
from collections import namedtuple, defaultdict # @launchit.collect
import json
import datetime
import pprint
import re
import pickle
import dataclasses # @launchit.collect
from dataclasses import dataclass # @launchit.collect
import IPython
from enum import Flag, StrEnum, auto # @launchit.collect
import sqlite3

from tqdm.notebook import tqdm

project_root_path = '${PROJECT_ROOT_PATH}' # @launchit.collect
# @launchit.disable
project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]
# @launchit.stop

sys.path.append(os.path.join(project_root_path, 'lib')) # @launchit.collect
from utils import * # @launchit.collect
from logging_utils import *
from image_utils import *
from artifact_registry import *
from torch_helpers import *
import launchit # @launchit.disable
import optuna_multiprocessing  # @launchit.collect
from hp_utils import *
from metrics_collector import RmqSummaryWriter
from autoincrement import Autoincrement

# Init

In [4]:
ArrayUtils.init()
LOG = Logging.get()
RNG = np.random.default_rng()

## Rename com.develorium.neurovision -> com.develorium.neurolab

In [13]:
assert False

groups = [
    'com.develorium.neurovision.10_denoise',
    'com.develorium.neurovision.11_cnn',
    'com.develorium.neurovision.12_rnn',
    'com.develorium.neurovision.13_gan',
    'com.develorium.neurovision.14_embedding',
    'com.develorium.neurovision.15_transformer',
    'com.develorium.neurovision.16_hybrid',
]

for group in tqdm(groups, desc='Group'):
    target_group = group.replace('neurovision', 'neurolab')
    ar_to = ArtifactRegistry(target_group)
    ar = ArtifactRegistry(group)
    models = ar.list_components()
    old_log_level = LOG.set_log_level('all', logging.ERROR)
    
    try:
        for model in tqdm(models, desc=f'Model {group.replace('com.develorium.neurovision.', '')}', leave=False):
            assets = ar.get_assets(model['name'], model['version'])
            ar_to.register_component(model['name'], model['version'])
        
            for asset in tqdm(assets, desc=f'Asset {model['name']}:{model['version']}', leave=False):
                ext = asset['maven2']['extension']
                
                if any(map(lambda x: ext.endswith(x), ('pom', '.md5', '.sha256', '.sha512', '.sha1'))):
                    continue
                    
                content = ar.get_asset_content(model['name'], model['version'], asset['maven2']['extension'], asset['maven2'].get('classifier', ''))
                
                with io.BytesIO(content) as b:
                    ar_to.attach_asset(
                        comp_name=model['name'], 
                        comp_version=model['version'], 
                        asset_ext=asset['maven2']['extension'], 
                        asset_classifier=asset['maven2'].get('classifier', ''),
                        asset=b)
    finally:
        LOG.set_log_level('all', old_log_level)

Group:   0%|          | 0/6 [00:00<?, ?it/s]

Model 11_cnn:   0%|          | 0/591 [00:00<?, ?it/s]

Asset s4_cnn_01:1:   0%|          | 0/25 [00:00<?, ?it/s]

Asset s4_cnn_01:7:   0%|          | 0/30 [00:00<?, ?it/s]

Asset s4_cnn_01:6:   0%|          | 0/30 [00:00<?, ?it/s]

...

Asset 15d_predict_noncausal_02:1:   0%|          | 0/20 [00:00<?, ?it/s]

Asset 15f_combo_noncausal_01:2:   0%|          | 0/5 [00:00<?, ?it/s]

Model 16_hybrid:   0%|          | 0/2 [00:00<?, ?it/s]

Asset 16e_hybrid_01:1:   0%|          | 0/25 [00:00<?, ?it/s]

Asset 16e_hybrid_01:2:   0%|          | 0/25 [00:00<?, ?it/s]